# Conversational RAG with LangChain, ChromaDB & Google Gemini
Welcome to your interactive RAG sandbox powered by Google Gemini and LangChain!

In this notebook, we test every piece of the pipeline step-by-step:
1. **Core Imports (`os`, `langchain`)**
2. **API Keys & Environment Setup** (Gemini + LangSmith Tracing)
3. **Quick Test: Say Hello with Gemini!**
4. **Document Ingestion** (`data/sample_docs/company_ai_policy.txt`)
5. **Text Chunking** (`RecursiveCharacterTextSplitter`)
6. **Embeddings & Vector Storage** (`models/gemini-embedding-001` & `ChromaDB`)
7. **Similarity Search & Relevance Scoring**
8. **Basic LCEL RAG Chain** (`gemini-3.5-flash`)
9. **Conversational Memory & History-Aware Retriever**

## Step 1: Import Core Modules (`os` and `langchain`)

In [ ]:
import os
import langchain

print(f"✅ os module loaded successfully.")
print(f"✅ LangChain version: {langchain.__version__}")

## Step 2: Load Environment Variables (Gemini & LangSmith)
Load `GOOGLE_API_KEY` and LangSmith tracing settings from `.env`.

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env in project root
load_dotenv(dotenv_path="../.env")

gemini_key = os.getenv("GOOGLE_API_KEY")
if not gemini_key or gemini_key == "your_gemini_api_key_here":
    print("⚠️ Notice: Please paste your GOOGLE_API_KEY into the .env file in the root folder.")
else:
    print(f"✅ Google Gemini API key loaded: {gemini_key[:8]}...")

langsmith_key = os.getenv("LANGSMITH_API_KEY")
tracing_enabled = os.getenv("LANGSMITH_TRACING")
print(f"🔍 LangSmith Tracing Active: {tracing_enabled} (Project: {os.getenv('LANGSMITH_PROJECT')})")

## Step 3: Quick Test — Say Hello using Gemini Model!
Let's test our connection by asking Gemini to say hello.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.7)

# Configure explicit SystemMessage and HumanMessagePromptTemplate
system_message = SystemMessage(
    content="You are a helpful and knowledgeable AI assistant. Provide concise and accurate answers."
)
human_template = HumanMessagePromptTemplate.from_template(
    "What is the capital of {country}? Give 1 interesting fact about it."
)

# Combine into ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([system_message, human_template])

# LCEL Pipeline: Prompt -> LLM -> Output Parser
chain = prompt | llm | StrOutputParser()

# Invoke dynamically with variables
response = chain.invoke({"country": "India"})

print("🤖 Gemini Response:")
print(response)


## Step 4: Load Documents (PDF, DOCX & TXT)
Load documents from both `document/` and `data/sample_docs/` using `PyPDFLoader`, `Docx2txtLoader`, and `TextLoader`.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

def load_document(file_path: str):
    """Load PDF, DOCX, or TXT file based on file extension."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext in [".docx", ".doc"]:
        loader = Docx2txtLoader(file_path)
    elif ext in [".txt", ".md"]:
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        return []
    return loader.load()

# Directories containing our policy files
document_dirs = ["../document", "../data/sample_docs"]
docs = []

for folder in document_dirs:
    if os.path.exists(folder):
        for file_name in os.listdir(folder):
            full_path = os.path.join(folder, file_name)
            if os.path.isfile(full_path) and file_name.endswith((".pdf", ".docx", ".txt", ".md")):
                loaded = load_document(full_path)
                docs.extend(loaded)
                print(f"📄 Loaded '{file_name}' ({folder}): {len(loaded)} page/section(s)")

print(f"\n📊 Total document parts/pages loaded: {len(docs)}")
if docs:
    print(f"Sample Metadata from first doc: {docs[0].metadata}")
    print("\n--- First 250 characters preview ---")
    print(docs[0].page_content[:250].strip())


## Step 5: Text Chunking (`RecursiveCharacterTextSplitter`)
Split the document into smaller semantic chunks with an overlap.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split documents into semantic chunks with overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(docs)
print(f"✅ Successfully split {len(docs)} documents into {len(chunks)} chunks!\n")

for i, chunk in enumerate(chunks[:2]):
    print(f"--- Chunk #{i+1} ({len(chunk.page_content)} characters) ---")
    print(f"Source: {chunk.metadata.get('source')} (Page: {chunk.metadata.get('page', 'N/A')})")
    print(chunk.page_content.strip())
    print("-" * 50)


## Step 6: Embeddings & Vector Store (Gemini + ChromaDB)
Embed chunks with `GoogleGenerativeAIEmbeddings` (`models/gemini-embedding-001`) and persist in ChromaDB.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize Google Gemini embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
persist_directory = "../data/chroma_db"

# Create and persist ChromaDB vector store from document chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="gemini_company_policy"
)

print(f"✅ Successfully indexed {len(chunks)} chunks into ChromaDB at '{persist_directory}'!")
print(f"Total vectors in collection: {vectorstore._collection.count()}")


## Step 7: Similarity Search (Retrieval)
Perform semantic similarity search on ChromaDB.

In [ ]:
# Test retrieval on our newly ingested company documents
query = "How many paid annual leave days do employees get?"
retrieved_results = vectorstore.similarity_search_with_relevance_scores(query, k=3)

print(f"🔍 Query: {query}\n")
for i, (doc, score) in enumerate(retrieved_results):
    print(f"Match #{i+1} (Relevance Score: {score:.4f}):")
    print(f"Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page', 'N/A')}")
    print(doc.page_content.strip())
    print("-" * 50)


## Step 8: Build LCEL RAG Chain with Gemini
Construct the RAG pipeline using `gemini-3.5-flash` and LCEL (`|`).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

prompt_template = """You are an expert AI assistant at NovaTech Solutions.
Answer the question strictly using the provided context. If the answer cannot be found in the context, say "I do not have enough information to answer that based on the provided documents."

Context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "What are the rules regarding Tier 3 confidential data?"
response = rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer:\n{response}")

## Step 9: Conversational RAG with Chat History
Handle multi-turn conversations and follow-up questions.

In [ ]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.messages import HumanMessage, AIMessage

# 1. Query reformulation prompt
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# 2. QA Answering Prompt
qa_system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know.\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# Multi-turn test
chat_history = []

# Turn 1
q1 = "What are the recommended chunking settings for RAG?"
res1 = conversational_rag_chain.invoke({"chat_history": chat_history, "input": q1})
print(f"User: {q1}")
print(f"Assistant: {res1['answer']}\n")

chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=res1["answer"])
])

# Turn 2 (Follow-up relying on history)
q2 = "Which vector stores are approved for that?"
res2 = conversational_rag_chain.invoke({"chat_history": chat_history, "input": q2})
print(f"User: {q2}")
print(f"Assistant: {res2['answer']}")